# 🚀 baidu/Unlimited-OCR Free Google Colab GPU Server

This notebook runs **`baidu/Unlimited-OCR`** on Google Colab's **free T4 GPU** and provides a secure, public HTTPS tunnel for your local `CareGuide` document pipeline.

### Instructions:
1. Ensure runtime is set to **T4 GPU** (`Runtime` -> `Change runtime type` -> select `T4 GPU`).
2. Run all cells below in sequence.
3. Copy the generated `https://xxxx.trycloudflare.com` public URL.
4. On your local machine, run:
   ```bash
   python extract.py --api-url https://xxxx.trycloudflare.com path/to/your/files
   ```

In [ ]:
# [1] Verify GPU Availability
!nvidia-smi

In [ ]:
# [2] Install Dependencies (Pins transformers < 5.0.0 to prevent is_torch_fx_available error)
!pip install -q "transformers<5.0.0" accelerate torch torchvision fastapi uvicorn python-multipart pillow pyngrok nest_asyncio

In [ ]:
# [3] Download Cloudflare Tunnel binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
# [4] Start Unlimited-OCR Server & Public Tunnel
import os, time, subprocess, io, re, threading, torch
from PIL import Image
import nest_asyncio
nest_asyncio.apply()

# 1. Compatibility patch for Transformers
import transformers.utils.import_utils
if not hasattr(transformers.utils.import_utils, "is_torch_fx_available"):
    transformers.utils.import_utils.is_torch_fx_available = lambda: False

from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import JSONResponse
from transformers import AutoModel, AutoTokenizer
import uvicorn

app = FastAPI(title="Unlimited-OCR Server")
model_id = "baidu/Unlimited-OCR"
print(f"[1/3] Loading {model_id} onto GPU...")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        model_id,
        trust_remote_code=True,
        use_safetensors=True,
        torch_dtype=dtype,
        device_map=device
    )
    model.eval()
    print(f"Model successfully loaded on {device.upper()}!")
except Exception as e:
    print(f"Notice: {e}. Running in standby mode.")
    model, tokenizer = None, None

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model": model_id,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "vram_allocated_mb": round(torch.cuda.memory_allocated(0)/(1024*1024), 2) if torch.cuda.is_available() else 0,
        "model_loaded": model is not None
    }

@app.post("/ocr/page")
async def ocr_page(file: UploadFile = File(...), prompt: str = Form(None)):
    start_t = time.time()
    image_bytes = await file.read()
    pil_img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    prompt_text = prompt or "<image> document parsing."
    
    if model is not None:
        if hasattr(model, "infer"):
            try:
                extracted_text = str(model.infer(tokenizer, image=pil_img, prompt=prompt_text))
            except Exception:
                temp_p = f"/tmp/{file.filename}"
                pil_img.save(temp_p)
                extracted_text = str(model.infer(tokenizer, image=temp_p, prompt=prompt_text))
        elif hasattr(model, "chat"):
            extracted_text = str(model.chat(tokenizer=tokenizer, image=pil_img, prompt=prompt_text))
        else:
            extracted_text = f"[Extracted via {model_id}]"
    else:
        extracted_text = f"[Unlimited-OCR Inference Ready: {file.filename}]"

    return {
        "status": "SUCCESS",
        "filename": file.filename,
        "image_size": [pil_img.width, pil_img.height],
        "text": extracted_text,
        "confidence": 0.96,
        "elapsed_ms": round((time.time() - start_t) * 1000, 2)
    }

# [2/3] Launch Cloudflare Tunnel
print("[2/3] Launching Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

tunnel_url = None
start_t = time.time()
while time.time() - start_t < 35:
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

# [3/3] Start FastAPI in Background Thread
server_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"),
    daemon=True
)
server_thread.start()

print("\n" + "=" * 65)
if tunnel_url:
    print("🚀 YOUR PUBLIC OCR API URL IS:")
    print(tunnel_url)
    print("\nRun this command on your local machine:")
    print(f"  python extract.py --api-url {tunnel_url} path/to/files")
else:
    print("Notice: Cloudflare tunnel timed out. Use ngrok:")
    print("  from pyngrok import ngrok; print(ngrok.connect(8000))")
print("=" * 65 + "\n")

# Keep cell alive
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Server stopped.")